# Apnea detection: dilated residual 1D CNN

Trains, validates and tests the network on the CinC Challenge 2018 database (PhysioNet): 994
overnight recordings from 994 people at Massachusetts General Hospital, Boston, with a 200 Hz
ECG. The people are split into 796 for training, 99 for validation and 99 for testing.

**Disk:** the Dataset cell downloads about 130 GB, since each night's ECG comes in one file with
its 12 other channels, and keeps about 26 GB: every night's ECG and labels in `data/nights/`
(about 20 GB), the arrays the model reads in `data/processed/` (about 5 GB). Downloaded files are
deleted once read.

**Run time:** the download takes 1 to 2 hours at 20 MB/s. The reference run trained on a T4 GPU
in Google Colab, where early stopping ended training after 24 epochs. Every cell skips what is
already done, and an interrupted Training cell resumes at its last finished epoch.

## Setup

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
# Hides TensorFlow's C++ INFO and WARNING lines (the converter prints pages of
# them). Must be set BEFORE tensorflow is imported; errors still show.

import csv                       # history.csv: one line per epoch, written during training
import hashlib                   # SHA-256 of every download, checked against PhysioNet's list
import http.client               # HTTPException: a transfer cut off mid-way
import json                      # history.json, the split lists, test_results.json
import random                    # Python's own generator, seeded below with the others
import shutil                    # deletes a night's downloaded files once they are read
import time                      # waits between two download attempts
import urllib.request            # downloads from PhysioNet
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed   # several nights at a time
from fractions import Fraction   # the resampling ratio, 200 Hz -> 100 Hz is 1/2
from pathlib import Path         # paths that work the same on macOS, Linux and Windows

import matplotlib.pyplot as plt  # training curves
import numpy as np               # arrays: the ECG windows, labels and predictions
import tensorflow as tf          # model, training, TFLite conversion
import wfdb                      # PhysioNet's library: reads recordings and their annotations
from scipy.signal import butter, filtfilt, iirnotch, resample_poly, sosfiltfilt   # resampling, filters
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve   # test scores, ROC curve
from tensorflow.keras import layers                    # the building blocks of the network
from tensorflow.keras.models import Model              # the functional-API model class

tf.get_logger().setLevel("ERROR")
# The same for TensorFlow's Python-side messages. "INFO:tensorflow:Assets written".
warnings.filterwarnings("ignore", message=".*tf.lite.Interpreter is deprecated.*")
# tf.lite.Interpreter still works; TensorFlow only announces that it will move
# to the separate LiteRT package.

KEEP_RAW = False
# True: the downloaded files (all 13 channels, about 130 GB) stay in
# data/cinc2018/ after they are read.

# ── folders ──────────────────────────────────────────────────────────────────
ROOT = Path.cwd().parent if Path.cwd().name == "training" else Path.cwd()
# The project root, whichever folder the kernel starts in. VS Code starts a
# notebook's kernel in the notebook's own folder (training/), a terminal-launched
# Jupyter usually at the root; both resolve to the same place.

RAW_DIR = ROOT / "data" / "cinc2018"         # PhysioNet files, as downloaded, while they are read
NIGHTS_DIR = ROOT / "data" / "nights"        # one .npz per night: ECG in mV at 200 Hz, minute labels
PROCESSED_DIR = ROOT / "data" / "processed"  # one-minute windows as .npy arrays, ready to train on
RESULTS_DIR = ROOT / "results"               # trained model, history, figures, TFLite files, scores
for folder in (RAW_DIR, NIGHTS_DIR, PROCESSED_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
    # parents=True creates data/ too if needed; exist_ok=True makes it a no-op
    # when the folder is already there, so the cell can be re-run freely.

# ── signal constants ─────────────────────────────────────────────────────────
FS = 100                    # the model's sampling rate; every night is resampled to it
WINDOW_SAMPLES = 60 * FS    # one label per minute
# The expert's apnea and hypopnea events are turned into one label per
# minute, so one minute is the unit: one window of 6000 samples = one label =
# one prediction.

# ── output files ─────────────────────────────────────────────────────────────
MODEL_PATH = RESULTS_DIR / "model.keras"             # the trained Keras model, best validation epoch
BEST_PATH = RESULTS_DIR / "best.keras"               # rewritten at every better epoch
BACKUP_DIR = RESULTS_DIR / "backup"                  # the last finished epoch, to resume from
LOG_PATH = RESULTS_DIR / "history.csv"               # one line per epoch, across interruptions
HISTORY_PATH = RESULTS_DIR / "history.json"          # loss and metrics for every epoch
CURVES_PATH = RESULTS_DIR / "training_curves.png"    # the four-panel figure
TEST_PATH = RESULTS_DIR / "test_results.json"        # the Test set cell's scores

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
# Weight initialisation, dropout masks and the per-epoch shuffle all draw
# random numbers. Seeding all three generators makes a rerun on the same
# machine and library versions produce the same model, so a change in the
# results can be attributed to a change in the code, not to chance.

# ── figures ──────────────────────────────────────────────────────────────────
BLUE, CORAL = "#0184FF", "#ff6a6a"     # the series colours, in this order
INK, INK_2, MUTED, RULE, SURFACE = "#8A8A8A", "#52606d", "#9aa5b1", "#e8ecef", "#ffffff"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": RULE, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 11.5, "axes.titleweight": "bold", "axes.titlepad": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": RULE, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[BLUE, CORAL]),
    "xtick.color": INK_2, "ytick.color": INK_2, "xtick.major.size": 0, "ytick.major.size": 0,
    "font.size": 10, "legend.frameon": False, "lines.linewidth": 2,
    "savefig.dpi": 150, "savefig.bbox": "tight"})
# One look for every figure: a light surface, with the axes and grid in light
# grey behind the data. A line drawn without a colour takes BLUE, the next one CORAL.

## Dataset

For every night: download its three files from PhysioNet's copy on Amazon S3 (physionet.org as a
fallback), check each against PhysioNet's SHA-256 list, read the ECG in mV and the expert's
events, and save one `data/nights/<night>.npz`:

- `ecg`: the ECG in mV, at 200 Hz
- `fs`: that rate
- `labels`: one value per minute: 1 an apnea or hypopnea in the minute, 0 none, -1 not scored

Nights already saved are skipped. A night that fails is listed at the end; running the cell again
retries it.

In [ ]:
# ── where the files are ──────────────────────────────────────────────────────
URLS = ["https://physionet-open.s3.amazonaws.com/challenge-2018/1.0.0/"] * 3 + [
    "https://physionet.org/files/challenge-2018/1.0.0/"]
# Tried in this order until one works: PhysioNet's copy on Amazon S3 three
# times, then physionet.org.


def fetch_text(name: str) -> str:
    problem = None
    for attempt, base in enumerate(URLS):
        if attempt:
            time.sleep(5 * attempt)
        try:
            with urllib.request.urlopen(base + name, timeout=60) as response:
                return response.read().decode()
        except (OSError, http.client.HTTPException) as error:
            problem = error
    raise OSError(f"{name}: {problem}")


# ── which nights, and which set each one belongs to ─────────────────────────
record_list = RAW_DIR / "RECORDS"
if not record_list.exists():
    record_list.write_text(fetch_text("RECORDS"))
RECORDS = [line.split("/")[1] for line in record_list.read_text().split() if line.startswith("training/")]
# PhysioNet's list of nights, downloaded once. Only the training/ half has
# public labels: 994 nights of 994 different people.


def files(record: str) -> list[str]:
    return [f"training/{record}/{record}{ext}" for ext in (".hea", ".mat", ".arousal")]
    # .hea      text header: sampling rate, length, scaling of each signal
    # .mat      the samples of all 13 signals, interleaved
    # .arousal  the expert's annotations: sleep stages, apneas, hypopneas, arousals


def assign_splits() -> dict[str, str]:
    people = sorted(RECORDS)
    order = np.random.default_rng(SEED).permutation(len(people))
    n_val = n_test = max(1, round(0.1 * len(people)))
    return {people[i]: ("val" if rank < n_val else "test" if rank < n_val + n_test else "train")
            for rank, i in enumerate(order)}
    # The people shuffled with SEED: 10 % validation, 10 % test, the rest
    # training.


SPLIT = assign_splits()
print(f"{len(RECORDS)} nights  " + "  ".join(
    f"{s} {list(SPLIT.values()).count(s)}" for s in ("train", "val", "test")))
# The split is BY PERSON, never by minute: every minute of one person lands
# in the same set. Splitting minutes at random puts the same heart on both
# sides, and the network learns to recognise the patient instead of the
# apnea.


# ── download ─────────────────────────────────────────────────────────────────
def download(name: str, dest: Path) -> None:
    problem = None
    for attempt, base in enumerate(URLS):
        if attempt:
            time.sleep(5 * attempt)
        part, digest = dest.with_name(dest.name + ".part"), hashlib.sha256()
        try:
            with urllib.request.urlopen(base + name, timeout=60) as response, open(part, "wb") as f:
                while chunk := response.read(1 << 20):
                    f.write(chunk)
                    digest.update(chunk)
        except (OSError, http.client.HTTPException) as error:
            problem = error
            continue
        if digest.hexdigest() == CHECKSUMS[name]:
            part.replace(dest)
            return
        problem = "the download does not match PhysioNet's checksum"
    raise OSError(f"{name}: {problem}")
    # Written to <name>.part in 1 MB pieces while the checksum is computed, and
    # renamed only when complete and correct. Each failed attempt waits 5 s
    # longer than the one before.


def night_files(record: str) -> Path:
    """The night's folder, after downloading the files it is missing."""
    folder = RAW_DIR / record
    folder.mkdir(parents=True, exist_ok=True)
    for name in files(record):
        if not (folder / Path(name).name).exists():
            download(name, folder / Path(name).name)
    return folder


# ── per-minute labels ────────────────────────────────────────────────────────
EVENTS = ("resp_obstructiveapnea", "resp_centralapnea", "resp_mixedapnea", "resp_hypopnea")
STAGES = ("W", "N1", "N2", "N3", "R")


def minute_labels(n_minutes: int, fs: float, events, scored) -> np.ndarray:
    """Events as (start, end) sample pairs -> per minute: 1 if one overlaps it, 0 if not, -1 not scored."""
    labels = np.full(n_minutes, -1, dtype=np.int8)
    first, last = int(np.ceil(scored[0] / (60 * fs))), int(scored[1] // (60 * fs))
    labels[first:min(last, n_minutes)] = 0
    for start, end in events:
        a, b = int(start // (60 * fs)), int((end - 1) // (60 * fs))
        for m in range(max(a, 0), min(b, n_minutes - 1) + 1):
            if labels[m] == 0:
                labels[m] = 1
    return labels
    # A minute is an apnea minute when an apnea or a hypopnea covers any part
    # of it. scored = (first, last) sample of the scored period; minutes not
    # wholly inside it stay -1.


def read_night(folder: Path, record: str):
    path = str(folder / record)
    rec = wfdb.rdrecord(path, channels=[12])
    assert rec.sig_name == ["ECG"], f"{record}: channel 12 is {rec.sig_name}"
    ecg, fs = rec.p_signal[:, 0], rec.fs
    ann = wfdb.rdann(path, "arousal")
    stages = [s for s, note in zip(ann.sample, ann.aux_note) if note in STAGES]
    opened, events = {}, []
    for sample, note in zip(ann.sample, ann.aux_note):
        if note.startswith("(") and note[1:] in EVENTS:
            opened[note[1:]] = sample
        elif note.endswith(")") and note[:-1] in opened:
            events.append((opened.pop(note[:-1]), sample))
    scored = (stages[0], stages[-1] + 30 * fs) if stages else (0, 0)
    return ecg, fs, minute_labels(len(ecg) // int(60 * fs), fs, events, scored)
    # Channel 12 of the 13 is the ECG, in mV. In the .arousal file "(name"
    # opens an event and "name)" closes it; W, N1, N2, N3 and R mark the 30 s
    # sleep stages, which span the scored period.


def prepare(record: str) -> None:
    folder = night_files(record)
    ecg, fs, labels = read_night(folder, record)
    target = NIGHTS_DIR / f"{record}.npz"
    part = target.with_name(target.name + ".part")
    with open(part, "wb") as f:
        np.savez(f, ecg=np.asarray(ecg, dtype=np.float32), fs=float(fs), labels=labels,
                 samples=len(ecg))
    part.replace(target)
    if not KEEP_RAW:
        shutil.rmtree(folder)
    # Written under another name and renamed once complete, so an interrupted
    # run never leaves half a night behind.


todo = [r for r in RECORDS if not (NIGHTS_DIR / f"{r}.npz").exists()]
if todo:
    CHECKSUMS = {name: digest for digest, name in
                 (line.split() for line in fetch_text("SHA256SUMS.txt").splitlines() if line.strip())}
    # Each line is "<sha256> <path within the database>".
failed = []
with ThreadPoolExecutor(max_workers=8) as pool:
    jobs = {pool.submit(prepare, r): r for r in todo}
    for done, job in enumerate(as_completed(jobs), start=1):
        try:
            job.result()
        except Exception as error:
            failed.append((jobs[job], str(error)))
        if done % 50 == 0 or done == len(jobs):
            print(f"  {done}/{len(jobs)} nights read", flush=True)
# Eight nights are downloaded and read side by side.

print(f"saved: {sum((NIGHTS_DIR / f'{r}.npz').exists() for r in RECORDS)}/{len(RECORDS)} nights")
for r, error in failed:
    print(f"  failed: {r}: {error}")
if failed:
    print("Run this cell again to retry the failed nights.")

## Windows

Every night through the preprocessing of the inference service
(`services/inference/source/pipeline.py`): resample to 100 Hz, band-pass 0.5 to 40 Hz, 50 Hz
notch, clip to +-5 mV, cut into one-minute windows, z-score each window. Only labelled minutes are
kept. One set of arrays per split (`train`, `val`, `test`) in `data/processed/`, stored as
float16:

- `X_<split>.npy` (minutes, 6000), `y_<split>.npy` (minutes,)
- `night_<split>.npy` (minutes,): the index of the window's night in `nights_<split>.json`

The arrays are rebuilt only when the nights or the split change.

In [ ]:
EPS = 1e-6
CLIP_MV = 5.0


def resample_to_model_rate(ecg: np.ndarray, fs: float) -> np.ndarray:
    if fs == FS:
        return ecg
    ratio = Fraction(FS / fs).limit_denominator(1000)
    return resample_poly(ecg, ratio.numerator, ratio.denominator)
    # Polyphase resampling, with its own anti-aliasing low-pass: 200 Hz -> 100 Hz
    # keeps one sample in two, after removing what 100 Hz cannot hold.


def clean_ecg(x: np.ndarray) -> np.ndarray:
    """A whole night of raw ECG (mV) -> filtered and clipped, same length."""
    x = sosfiltfilt(butter(4, [0.5, 40], btype="band", fs=FS, output="sos"), x)   # baseline wander, noise
    # Butterworth band-pass keeping 0.5-40 Hz:
    #   below 0.5 Hz  baseline wander, from breathing and body movement
    #   above 40 Hz   muscle activity and electronic noise
    # output="sos" (second-order sections) keeps the filter numerically
    # stable. sosfiltfilt runs it forward then backward, so the two phase
    # shifts cancel and the heartbeats do not move in time. That needs the
    # whole recording at once, which is fine here, offline.

    x = filtfilt(*iirnotch(50, Q=30, fs=FS), x)                                  # mains
    # Narrow notch at 50 Hz, the mains frequency (Europe, Morocco). At 100 Hz
    # sampling, 50 Hz is exactly the highest frequency the signal can hold,
    # and the band-pass above already removes it; the notch is kept so the
    # windows stay identical to the ones the model was trained on.

    x = np.nan_to_num(x, nan=np.nanmedian(x))
    # Safeguard: replaces any missing sample with the night's median. The
    # median, not the mean, because R peaks and artifacts pull the mean.

    return np.clip(x, -CLIP_MV, CLIP_MV)                                         # artifacts, mV
    # A real ECG stays well within +-5 mV; beyond that is an artifact (an
    # electrode moving or coming off). Clipping bounds it instead of deleting
    # it, so no sample is removed and the minute grid never shifts.


def to_windows(ecg_clean: np.ndarray) -> np.ndarray:
    """Whole minutes from the start of the recording -> (N, 6000) float32, each z-scored."""
    n = len(ecg_clean) // WINDOW_SAMPLES
    w = ecg_clean[:n * WINDOW_SAMPLES].astype(np.float32).reshape(n, WINDOW_SAMPLES)
    w = (w - w.mean(axis=1, keepdims=True)) / (w.std(axis=1, keepdims=True) + EPS)
    return np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    # Z-score: subtract the minute's mean, divide by its standard deviation.
    # Each minute on its own: absolute amplitude reflects electrode placement,
    # not breathing. EPS avoids a division by zero on a flat minute.
# The three functions above compute what services/inference/source/pipeline.py
# computes: the device and the service must feed the model the same windows.


def night_windows(record: str):
    night = np.load(NIGHTS_DIR / f"{record}.npz")
    w = to_windows(clean_ecg(resample_to_model_rate(night["ecg"].astype(np.float64), float(night["fs"]))))
    labels = night["labels"][:len(w)]
    keep = np.flatnonzero(labels >= 0)
    return w[keep].astype(np.float16), labels[keep].astype(np.float32)
    # Only the minutes the expert scored. Never dropped for signal quality: a
    # real device has to answer for the noisy minutes too.


def kept_minutes(record: str) -> int:
    night = np.load(NIGHTS_DIR / f"{record}.npz")
    fs, samples = float(night["fs"]), int(night["samples"])
    ratio = Fraction(FS / fs).limit_denominator(1000)
    n = -(-samples * ratio.numerator // ratio.denominator) // WINDOW_SAMPLES
    return int((night["labels"][:n] >= 0).sum())
    # The windows night_windows() keeps, counted from the labels and the length
    # alone: resample_poly returns ceil(samples * up / down) samples.


def split_files(split: str) -> list[Path]:
    return [PROCESSED_DIR / name for name in (f"X_{split}.npy", f"y_{split}.npy",
                                              f"night_{split}.npy", f"nights_{split}.json")]


SPLITS = ("train", "val", "test")
available = [r for r in RECORDS if (NIGHTS_DIR / f"{r}.npz").exists()]
config = {"nights": [[r, SPLIT[r]] for r in available], "fs": FS, "version": 1}
config_path = PROCESSED_DIR / "config.json"

if config_path.exists() and json.loads(config_path.read_text()) == config:
    print("windows: already built for these nights and this split")
else:
    config_path.unlink(missing_ok=True)
    for split in SPLITS:
        for f in split_files(split):
            f.unlink(missing_ok=True)
    for split in SPLITS:
        nights = [r for r, s in config["nights"] if s == split]
        sizes = [kept_minutes(r) for r in nights]
        offsets = np.concatenate([[0], np.cumsum(sizes)])
        x_path, y_path, night_path, list_path = split_files(split)
        X = np.lib.format.open_memmap(x_path, mode="w+", dtype=np.float16,
                                      shape=(int(offsets[-1]), WINDOW_SAMPLES))
        y = np.zeros(int(offsets[-1]), dtype=np.float32)

        def fill(i):
            w, labels = night_windows(nights[i])
            assert len(labels) == sizes[i], f"{nights[i]}: {len(labels)} windows, expected {sizes[i]}"
            X[offsets[i]:offsets[i + 1]] = w
            y[offsets[i]:offsets[i + 1]] = labels

        with ThreadPoolExecutor(max_workers=4) as pool:
            for done, _ in enumerate(pool.map(fill, range(len(nights))), start=1):
                if done % 100 == 0 or done == len(nights):
                    print(f"  {split}: {done}/{len(nights)} nights", flush=True)
        X.flush()
        del X
        np.save(y_path, y)
        np.save(night_path, np.repeat(np.arange(len(nights), dtype=np.int32), sizes))
        list_path.write_text(json.dumps(nights))
    config_path.write_text(json.dumps(config))
    # The window count of every night is known before any window is computed,
    # so X is created at its final size directly on disk (open_memmap) and each
    # night writes its own rows. config.json is written last: an interrupted
    # build is redone from the start.

for split in SPLITS:
    y = np.load(PROCESSED_DIR / f"y_{split}.npy")
    nights = json.loads((PROCESSED_DIR / f"nights_{split}.json").read_text())
    print(f"{split:5s} {len(y):7d} minutes from {len(nights):4d} nights, {y.mean():.1%} apnea")

## Hyperparameters

In [ ]:
# ── stem: shrink the minute before the expensive part ────────────────────────
STEM = (
    #  filt  ker  str  pool
    (16,     15,   2,   1),
    (24,      7,   1,   2),
    (40,      7,   1,   2),
    (64,      5,   1,   2),
)
# Four convolution layers, one row each:
#   filt  number of filters (feature channels) the layer produces
#   ker   kernel length in samples: how much signal each filter looks at
#   str   stride: 2 keeps every second output, halving the length
#   pool  max-pooling factor after the layer (1 = none)
# Length through the stem: 6000 -> 3000 -> 1500 -> 750 -> 375 timesteps,
# one step every 16 samples (160 ms). The first kernel, 15 samples
# = 150 ms, is about one QRS complex wide, so the first layer can already see
# a whole heartbeat. The last row's 64 filters are replaced by TRUNK_FILTERS
# in build_model(), so the stem hands the trunk exactly the width it uses.

# ── trunk: residual blocks that look far back in time ────────────────────────
TRUNK_FILTERS = 32                    # channels in every residual block
TRUNK_KERNEL = 3                      # kernel length of the dilated convolutions
DILATIONS = (1, 2, 4, 8, 16, 32, 64)  # one block per value
# A dilated convolution with dilation d reads every d-th step. Doubling d at
# each block makes the view grow exponentially while the parameter count
# grows only linearly: after seven blocks one output sees 8225 input samples
# (82 s), more than the whole 60 s minute. build_model() checks this.
SE_RATIO = 8                          # squeeze-excite bottleneck: 32 channels -> 4 -> 32
DROPOUT_TRUNK = 0.10                  # share of channels switched off per block, training only
DROPOUT_HEAD = 0.30                   # share of units switched off before the output
# Dropout fights overfitting: it keeps the network from memorising
# individuals. It is active during training only; at inference every unit
# is used.

# ── training ─────────────────────────────────────────────────────────────────
EPOCHS = 70               # upper bound; early stopping usually ends the run far sooner
BATCH_SIZE = 64           # minutes per gradient step
LEARNING_RATE = 1e-3      # Adam's usual starting step size
EARLY_STOP_PATIENCE = 12  # epochs without a better val_auc before training stops
REDUCE_LR_PATIENCE = 6    # epochs without a better val_auc before the learning rate is halved
# REDUCE_LR_PATIENCE < EARLY_STOP_PATIENCE on purpose: when validation stalls,
# the run first gets a smaller step size and a second chance to improve
# before it is stopped.
FLIP = True               # each training window multiplied by -1 with probability 0.5
# The sign of a recorded ECG depends on how the electrodes were placed. With
# FLIP the model sees every window both ways up; False keeps them as recorded.

## Data

In [ ]:
def load_split(split: str):
    """One split -> X (minutes, 6000) float16 read from disk on demand, y, night index, nights."""
    X = np.load(PROCESSED_DIR / f"X_{split}.npy", mmap_mode="r")
    y = np.load(PROCESSED_DIR / f"y_{split}.npy")
    night = np.load(PROCESSED_DIR / f"night_{split}.npy")
    assert len(X) == len(y) == len(night), f"{split}: arrays disagree"
    # One label per window. A mismatch means the files come from different
    # runs: stop rather than train on shifted labels.
    return X, y, night, json.loads((PROCESSED_DIR / f"nights_{split}.json").read_text())
    # mmap_mode="r": the array stays on disk and only the rows asked for are
    # read, so the 4 GB training array never has to fit in memory.


def batches(X, y, batch_size: int, shuffle: bool = False, flip: bool = False) -> tf.data.Dataset:
    def gather(index):
        index = np.sort(index)
        return X[index].astype(np.float32)[..., np.newaxis], y[index]

    ds = tf.data.Dataset.range(len(y))
    if shuffle:
        ds = ds.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).map(
        lambda index: tf.numpy_function(gather, [index], (tf.float32, tf.float32)),
        num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, t: (tf.ensure_shape(x, (None, WINDOW_SAMPLES, 1)), tf.ensure_shape(t, (None,))))
    if flip:
        ds = ds.map(lambda x, t: (x * tf.where(tf.random.uniform((tf.shape(x)[0], 1, 1)) < 0.5, -1.0, 1.0), t))
    return ds.prefetch(tf.data.AUTOTUNE)
    # The dataset holds window numbers; each batch of numbers is turned into
    # windows by reading those rows of X (sorted, so the disk is read in
    # order), widened to float32 and given the channel axis Conv1D expects:
    # (batch, 6000, 1). shuffle draws a new order of all window numbers every
    # epoch. flip multiplies each window by -1 or +1 at random. prefetch
    # prepares the next batch while the current one trains.


X_tr, y_tr, night_tr, nights_tr = load_split("train")     # the network learns from these
X_val, y_val, night_val, nights_val = load_split("val")   # never trained on, used to judge each epoch
# The test split is not loaded: it is kept for the final evaluation, once
# every training decision has been made.
print(f"train {X_tr.shape}  apnea {y_tr.mean():.1%}")
print(f"val   {X_val.shape}  apnea {y_val.mean():.1%}")
# The mean of a 0/1 label array is the share of apnea minutes.

## Building Blocks

In [ ]:
def receptive_field() -> tuple[int, int]:
    """How many input samples one trunk output can see, and how many samples one trunk step spans."""
    span, stride = 1, 1
    # span:   width of the input region seen so far, in samples
    # stride: distance in input samples between two neighbouring outputs
    for _filters, kernel, conv_stride, pool in STEM:
        span += (kernel - 1) * stride      # a kernel of k widens the view by k - 1 steps
        stride *= conv_stride              # a strided layer spaces later steps further apart
        if pool > 1:
            span += (pool - 1) * stride    # pooling widens the view like a kernel of size pool
            stride *= pool
    for dilation in DILATIONS:
        effective_kernel = 1 + (TRUNK_KERNEL - 1) * dilation   # kernel 3 at dilation d spans 2d + 1 steps
        span += 2 * (effective_kernel - 1) * stride            # two convolutions per residual block
    return span, stride


def squeeze_excite(x, ratio: int, name: str):
    """Channel attention: learn how much each feature channel matters for THIS minute."""
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling1D(name=f"{name}_se_squeeze")(x)
    # "Squeeze": average each channel over the whole minute -> one number per channel.
    se = layers.Dense(max(filters // ratio, 4), activation="relu", name=f"{name}_se_reduce")(se)
    # Bottleneck (32 -> 4): forces a compact summary of which channels are active.
    se = layers.Dense(filters, activation="sigmoid", name=f"{name}_se_expand")(se)
    # "Excite": back to one weight per channel, each between 0 and 1.
    se = layers.Reshape((1, filters), name=f"{name}_se_reshape")(se)
    # (batch, channels) -> (batch, 1, channels), so it broadcasts over every timestep.
    return layers.Multiply(name=f"{name}_se_scale")([x, se])
    # Every channel is scaled by its weight: useful features are kept,
    # irrelevant ones turned down, differently for every minute.


def conv_bn(x, filters: int, kernel: int, conv_name: str, bn_name: str,
            dilation: int = 1, strides: int = 1):
    """Convolution followed by batch normalisation: the pair used everywhere below."""
    x = layers.Conv1D(filters, kernel, strides=strides, padding="same", dilation_rate=dilation,
                      use_bias=False, name=conv_name)(x)
    # padding="same" keeps the length unchanged (unless strides > 1), so the
    # shortcut and the main path of a residual block can be added.
    # use_bias=False: the batch normalisation right after has its own shift
    # term, so a bias here would be a redundant parameter.
    return layers.BatchNormalization(name=bn_name)(x)
    # Re-centres and rescales each channel over the batch. It keeps the
    # activations in a stable range, which makes deep networks train faster
    # and less sensitively to the learning rate. On the device it is folded
    # into the convolution's weights and costs nothing.


def residual_block(x, filters: int, kernel: int, dilation: int, dropout: float, name: str):
    """Two dilated convolutions plus a shortcut: output = relu(input + learned correction)."""
    shortcut = x
    if x.shape[-1] != filters:
        shortcut = conv_bn(x, filters, 1, f"{name}_proj", f"{name}_proj_bn")
        # A 1x1 convolution only when the channel counts differ, so the two
        # paths can be added. Here the stem already outputs TRUNK_FILTERS
        # channels, so every shortcut is the plain identity.

    y = conv_bn(x, filters, kernel, f"{name}_conv1", f"{name}_bn1", dilation)
    y = layers.Activation("relu", name=f"{name}_relu1")(y)
    y = layers.SpatialDropout1D(dropout, name=f"{name}_drop")(y)
    # SpatialDropout1D switches off whole CHANNELS, not single samples.
    # Neighbouring ECG samples are so correlated that dropping one sample
    # hides nothing; dropping a whole channel forces the other channels to
    # carry the information too.
    y = conv_bn(y, filters, kernel, f"{name}_conv2", f"{name}_bn2", dilation)
    y = squeeze_excite(y, SE_RATIO, name)

    y = layers.Add(name=f"{name}_add")([shortcut, y])
    # The shortcut: the block only has to learn a correction to its input.
    # It also gives gradients a direct path back through all seven blocks, so
    # the early layers still learn.
    return layers.Activation("relu", name=f"{name}_out")(y)


def attention_pool(x, name: str = "attn"):
    """Weighted average over time, with the weights learned: where in the minute to look."""
    scores = layers.Conv1D(1, 1, name=f"{name}_score")(x)
    # One score per timestep: how relevant this moment of the minute looks.
    weights = layers.Softmax(axis=1, name=f"{name}_softmax")(scores)
    # Softmax over time: positive weights that sum to 1 across the 375 steps.
    pooled = layers.Dot(axes=1, name=f"{name}_pool")([x, weights])
    # Weighted sum over time: (batch, 375, 32) x (batch, 375, 1) -> (batch, 32, 1).
    return layers.Flatten(name=f"{name}_flat")(pooled)
    # An apnea shows up in a few seconds of the minute, typically the heart
    # rate swing when breathing restarts. A plain average dilutes those
    # seconds into the other fifty; attention lets the model weight them up.

## Model

In [ ]:
def build_model(input_length: int = WINDOW_SAMPLES) -> Model:
    """ECG minute (6000, 1) -> stem -> 7 residual blocks -> 3 poolings -> P(apnea)."""
    span, _stride = receptive_field()
    assert span >= input_length, (
        f"receptive field {span} samples ({span / FS:.2f} s) does not cover "
        f"the {input_length}-sample window")
    # If a change to STEM or DILATIONS leaves the view shorter than the
    # minute, the model could not relate its start to its end. Fail here,
    # loudly, instead of training a model that is blind to part of its input.

    inputs = layers.Input(shape=(input_length, 1), name="ecg")
    # The batch dimension is left free; Keras adds it.

    # ── stem ─────────────────────────────────────────────────────────────────
    x = inputs
    for i, (filters, kernel, conv_stride, pool) in enumerate(STEM, start=1):
        if i == len(STEM):
            filters = TRUNK_FILTERS
            # The last stem layer outputs the trunk's width directly, so no
            # projection is needed in the first residual block.
        x = conv_bn(x, filters, kernel, f"stem{i}_conv", f"stem{i}_bn", strides=conv_stride)
        x = layers.Activation("relu", name=f"stem{i}_relu")(x)
        if pool > 1:
            x = layers.MaxPooling1D(pool, name=f"stem{i}_pool")(x)
            # Keeps the strongest response in each group of `pool` steps and
            # halves the length: cheaper layers, and tolerance to small shifts.

    # ── trunk ────────────────────────────────────────────────────────────────
    for i, dilation in enumerate(DILATIONS, start=1):
        x = residual_block(x, TRUNK_FILTERS, TRUNK_KERNEL, dilation, DROPOUT_TRUNK,
                           name=f"block{i}_d{dilation}")
    # x is now (batch, 375, 32): 375 moments of the minute, 32 features each.

    # ── pooling: from a sequence to one vector per minute ───────────────────
    pooled = layers.Concatenate(name="pool_concat")([
        attention_pool(x),                                   # the moments the model finds relevant
        layers.GlobalAveragePooling1D(name="avg_pool")(x),   # the overall level across the minute
        layers.GlobalMaxPooling1D(name="max_pool")(x),       # the single strongest event
    ])
    # Three complementary summaries side by side: 3 x 32 = 96 numbers.

    # ── head: the decision ───────────────────────────────────────────────────
    y = layers.Dense(64, activation="relu", name="head_dense")(pooled)
    y = layers.Dropout(DROPOUT_HEAD, name="head_drop")(y)
    outputs = layers.Dense(1, activation="sigmoid", name="apnea_prob")(y)
    # One unit with a sigmoid: a number between 0 and 1, read as the
    # probability that this minute is apnea.

    return Model(inputs, outputs, name="apnea_dilated_cnn")


model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    # Adam adapts the step size of every weight to its own gradient history;
    # a robust default that needs little tuning.
    loss=tf.keras.losses.BinaryCrossentropy(),
    # The standard loss for a yes/no output. It punishes confident mistakes
    # hardest: predicting 0.99 for a normal minute costs far more than 0.6.
    metrics=["accuracy",
             tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Recall(name="recall"),
             tf.keras.metrics.Precision(name="precision")],
    # Reported every epoch, not optimised directly:
    #   accuracy   share of minutes classified right, at a 0.5 cut-off
    #   auc        how well apnea minutes are ranked above normal ones, at
    #              every cut-off at once: the metric the callbacks watch
    #   recall     share of apnea minutes caught (sensitivity), at 0.5
    #   precision  share of "apnea" answers that were right, at 0.5
)
# model.summary()

## Training

Besides early stopping and the learning-rate schedule, three callbacks keep the run on disk: the
last finished epoch (`results/backup/`), the best epoch so far (`results/best.keras`) and one line
per epoch (`results/history.csv`). If training is interrupted, run Setup, Hyperparameters, Data,
Building Blocks and Model again, then this cell: it continues from the last finished epoch,
and early stopping and the learning-rate schedule pick up their count from `history.csv`.
Without a backup, it starts a new run.

In [ ]:
class LearningRateLog(tf.keras.callbacks.Callback):
    """Writes the current learning rate into the history at the end of every epoch."""
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None:
            logs["lr"] = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
    # Keras does not record the learning rate by itself. With it in
    # history.json, the effect of ReduceLROnPlateau can be read back later.


def val_auc_so_far() -> list[float]:
    """val_auc of every epoch in history.csv, those before an interruption included."""
    if not LOG_PATH.exists():
        return []
    with open(LOG_PATH) as f:
        return [float(row["val_auc"]) for row in csv.DictReader(f) if row.get("val_auc")]


class Resumable:
    """Gives EarlyStopping and ReduceLROnPlateau back their count when fit() resumes."""
    restarts_count = False   # True: the count starts again from 0 once it reaches patience

    def on_train_begin(self, logs=None):
        super().on_train_begin(logs)
        best, wait = None, 0
        for value in val_auc_so_far():
            if best is None or value - self.min_delta > best:
                best, wait = value, 0
            else:
                wait += 1
                if self.restarts_count and wait >= self.patience:
                    wait = 0
        if best is not None:
            self.best, self.wait = best, wait
    # BackupAndRestore puts back the weights, the optimizer and the epoch
    # number, not these callbacks' state: Keras starts them with no best value
    # and a count of 0. This replays history.csv as Keras counts: best is the
    # highest val_auc so far, wait the number of epochs since it, or since the
    # last halving of the learning rate for ReduceLROnPlateau. Without a
    # history.csv, Keras' own start is kept.


class ResumableEarlyStopping(Resumable, tf.keras.callbacks.EarlyStopping):
    pass


class ResumableReduceLROnPlateau(Resumable, tf.keras.callbacks.ReduceLROnPlateau):
    restarts_count = True


if not BACKUP_DIR.exists():
    LOG_PATH.unlink(missing_ok=True)
    BEST_PATH.unlink(missing_ok=True)
    # A new run: the log and best epoch of an earlier run are removed. With a
    # backup, fit() resumes the interrupted run and both are kept.
done = val_auc_so_far()
if done:
    print(f"resuming after epoch {len(done)}: best val_auc {max(done):.4f} at epoch "
          f"{int(np.argmax(done)) + 1}")

history = model.fit(
    batches(X_tr, y_tr, BATCH_SIZE, shuffle=True, flip=FLIP),
    validation_data=batches(X_val, y_val, 512),
    # After every epoch the model is scored on the validation people. It
    # never trains on them, so val_* shows how it does on unseen people.
    epochs=EPOCHS,
    shuffle=False,
    # batches() already shuffles; Keras' own shuffle does not apply to a
    # tf.data pipeline and would only print a warning.
    callbacks=[
        LearningRateLog(),
        tf.keras.callbacks.BackupAndRestore(str(BACKUP_DIR)),
        # Saves model, optimizer and epoch number at the end of every epoch;
        # fit() starts from there when a backup exists, and deletes it when
        # training ends normally.
        tf.keras.callbacks.CSVLogger(str(LOG_PATH), append=True),
        tf.keras.callbacks.ModelCheckpoint(
            str(BEST_PATH), monitor="val_auc", mode="max", save_best_only=True,
            initial_value_threshold=max(done, default=None)),
        # Rewrites best.keras whenever val_auc beats every earlier epoch,
        # those before an interruption included (read back from history.csv).
        ResumableEarlyStopping(
            monitor="val_auc", mode="max", patience=EARLY_STOP_PATIENCE, verbose=1),
        # Stops once val_auc has not improved for 12 epochs. The Save cell then
        # loads best.keras: the BEST epoch, not the last one.
        ResumableReduceLROnPlateau(
            monitor="val_auc", mode="max", factor=0.5,
            patience=REDUCE_LR_PATIENCE, min_lr=1e-6, verbose=1),
        # Halves the learning rate after 6 epochs without improvement: smaller
        # steps can settle into a minimum that large steps kept jumping over.
        # Both callbacks watch AUC because it does not depend on a threshold;
        # the 0.5 behind accuracy is not the one used in deployment.
    ],
    verbose=1,   # one progress bar per epoch
)

## Save

In [ ]:
model = tf.keras.models.load_model(BEST_PATH)
model.save(MODEL_PATH)
# best.keras holds the epoch with the highest val_auc, across interruptions
# too; model.keras is that model under its usual name. Architecture and
# weights in one file.

with open(LOG_PATH) as f:
    rows = list(csv.DictReader(f))
record = {key: [float(row[key]) if row[key] not in ("", "NA") else float("nan") for row in rows]
          for key in rows[0] if key != "epoch"}
HISTORY_PATH.write_text(json.dumps(record, indent=1))
# history.csv as a dict of lists, one value per epoch for every loss and
# metric (train and val_), plus "lr". It lets the curves be redrawn, or two
# runs compared, without retraining.

print(f"saved {MODEL_PATH}")
print(f"saved {HISTORY_PATH}")

## Training curves

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
record = json.loads(HISTORY_PATH.read_text())
# Both read from disk, so this cell works after a kernel restart.

prob_val = model.predict(batches(X_val, y_val, 512), verbose=0).ravel()
# P(apnea) for every validation minute, from the restored model. .ravel()
# flattens (minutes, 1) to (minutes,).
epochs = np.arange(1, len(record["loss"]) + 1)      # 1, 2, ... for the x axis
best = int(np.argmax(record["val_auc"])) + 1        # the epoch whose weights were kept

fig, axes = plt.subplots(2, 2, figsize=(10.5, 7))
panels = [
    (axes[0, 0], "Loss", [("Train", "loss"), ("Validation", "val_loss")]),
    (axes[0, 1], f"AUC (best val {max(record['val_auc']):.4f} at epoch {best})",
     [("Train", "auc"), ("Validation", "val_auc")]),
    (axes[1, 0], "Validation precision & recall at 0.5",
     [("Precision", "val_precision"), ("Recall", "val_recall")]),
]
for ax, title, series in panels:
    for label, key in series:
        ax.plot(epochs, record[key], label=label)
    ax.axvline(best, color=MUTED, linestyle="--", linewidth=1)
    # The dashed line marks the restored epoch.
    ax.set(title=title, xlabel="Epoch")
    ax.legend()

fpr, tpr, _ = roc_curve(y_val, prob_val)
# The ROC curve: for every possible cut-off, the share of normal minutes
# wrongly flagged (false positive rate) against the share of apnea minutes
# caught (true positive rate).
ax = axes[1, 1]
ax.plot([0, 1], [0, 1], color=MUTED, linestyle="--", linewidth=1)
# The diagonal is what a coin flip would score.
ax.plot(fpr, tpr, label=f"AUC {roc_auc_score(y_val, prob_val):.4f}")
ax.set(title="Validation ROC", xlabel="False positive rate", ylabel="True positive rate")
ax.set_aspect("equal", adjustable="box")   # square, so the curve's shape is not distorted
ax.legend(loc="lower right")

fig.tight_layout()                  # stops titles and labels from overlapping
fig.savefig(CURVES_PATH)
plt.show()
print(f"saved {CURVES_PATH}")

## TensorFlow Lite conversion

Converts `results/model.keras` into two TFLite files:

- `model_int8.tflite`: fully int8, for TFLite Micro on the ESP32-S3. Weights, activations, input
  and output are int8; activation ranges are calibrated on 500 real training windows.
- `model_float32.tflite`: the same network unquantised, computing what `model.keras` computes.

Both share:

- **Fixed input shape `(1, 6000, 1)`**: a microcontroller allocates every tensor up front.
- **A SavedModel export first.** Converting the Keras 3 object directly leaves the weights as
  variables, and int8 calibration then fails.

Only needs the Setup, Hyperparameters and Data cells; it does not need retraining.

In [ ]:
import tempfile   # a scratch folder that deletes itself

TFLITE_PATH = RESULTS_DIR / "model_int8.tflite"
FLOAT32_PATH = RESULTS_DIR / "model_float32.tflite"

model = tf.keras.models.load_model(MODEL_PATH)
# Reloaded from disk, so this section works on its own after a kernel
# restart, without retraining.


def representative_dataset():
    """500 real training minutes, one at a time, for the converter to calibrate int8 ranges."""
    rng = np.random.default_rng(SEED)
    for i in rng.choice(len(X_tr), size=500, replace=False):
        yield [X_tr[i:i + 1].astype(np.float32)[..., np.newaxis]]
    # The converter runs these through the float model and records how large
    # every internal value gets. Those ranges set each tensor's int8 scale.
    # They must be REAL windows: random noise would give the wrong ranges and
    # cost accuracy. No labels are needed. X_tr[i:i + 1] keeps the batch
    # axis; the float16 window is widened to float32 and given its channel
    # axis: shape (1, 6000, 1).


with tempfile.TemporaryDirectory() as saved_model_dir:
    # A temporary folder, deleted with its contents when the block ends: the
    # SavedModel is only an intermediate step.
    model.export(saved_model_dir, format="tf_saved_model", verbose=False,
                 input_signature=[tf.TensorSpec([1, WINDOW_SAMPLES, 1], tf.float32, name="ecg")])
    # SavedModel: the network with its weights frozen as constants. Converting
    # the Keras 3 model object directly leaves the weights as variables, and
    # int8 calibration then fails. The input shape is fixed at one window of
    # 6000 samples: a microcontroller reserves all its memory up front and
    # cannot handle a variable batch size.

    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]      # turn quantisation on
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    # Only int8 operations are allowed. If any layer could not be quantised,
    # conversion fails instead of silently keeping a float operation the
    # microcontroller would run slowly or not at all.
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    # Input and output in int8 too: no float left anywhere in the model. The
    # device quantises the z-scored window before inference and turns the
    # int8 output back into a probability after.
    tflite_model = converter.convert()   # the model as bytes, in TFLite's flatbuffer format

    float32_model = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir).convert()
    # The same SavedModel converted with no optimisation set: weights,
    # activations, input and output all stay float32, so this file computes
    # what model.keras computes, with the same fixed (1, 6000, 1) input. It
    # takes the z-scored window and returns the probability directly, with no
    # quantise or dequantise step, and needs no calibration windows. It is the
    # larger file: every weight takes 4 bytes instead of 1.

TFLITE_PATH.write_bytes(tflite_model)
FLOAT32_PATH.write_bytes(float32_model)
print(f"saved {TFLITE_PATH} ({len(tflite_model) / 1024:.1f} KB)")
print(f"saved {FLOAT32_PATH} ({len(float32_model) / 1024:.1f} KB)")

## Test set

Scores `results/model.keras`, `results/model_float32.tflite` and `results/model_int8.tflite` on the
99 test people, who played no part in training. For each model the threshold is chosen on the
validation people, then applied once to the test set. The scores are saved to `results/test_results.json`.

Needs the Setup and Data cells; it does not need retraining.

In [ ]:
SMOOTH_WINDOW = 5                              # minutes averaged into each decision
THRESHOLD_GRID = np.arange(0.05, 0.96, 0.01)   # candidate thresholds, tried on validation
TFLITE_PATH = RESULTS_DIR / "model_int8.tflite"
FLOAT32_PATH = RESULTS_DIR / "model_float32.tflite"

X_te, y_te, night_te, nights_te = load_split("test")


def smooth(prob: np.ndarray, night: np.ndarray) -> np.ndarray:
    """Centred moving average over SMOOTH_WINDOW minutes, within each night separately."""
    out = np.empty_like(prob)
    starts = np.flatnonzero(np.diff(night, prepend=-1))
    for a, b in zip(starts, np.append(starts[1:], len(prob))):
        padded = np.pad(prob[a:b], SMOOTH_WINDOW // 2, mode="edge")
        out[a:b] = np.convolve(padded, np.ones(SMOOTH_WINDOW) / SMOOTH_WINDOW, mode="valid")
    return out
    # A night's windows are consecutive rows; a new night starts where the
    # night index changes. Apnea comes in runs of several minutes while the
    # model scores each minute alone; averaging neighbours removes isolated
    # spikes. np.pad repeats the edge values so the first and last minutes of
    # a night are averaged over a full window. Night by night, so one person's
    # night is never averaged with the next person's.


keras_model = tf.keras.models.load_model(MODEL_PATH)


def predict_keras(X) -> np.ndarray:
    return keras_model.predict(batches(X, np.zeros(len(X), np.float32), 512), verbose=0).ravel()


def tflite_predictor(path: Path):
    """A function that runs the .tflite file at path on each window in turn, as the device does."""
    interpreter = tf.lite.Interpreter(
        model_path=str(path),
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
    # Plain built-in kernels: the desktop XNNPACK delegate cannot prepare the int8
    # graph, and the reference kernels are closer to what TFLite Micro runs.
    interpreter.allocate_tensors()
    inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
    quantized = inp["dtype"] == np.int8
    in_scale, in_zero = inp["quantization"]
    out_scale, out_zero = out["quantization"]
    # int8 holds whole numbers from -128 to 127. Each tensor maps them to real
    # values with two constants:  real = scale x (q - zero_point). The float32
    # file reports (0.0, 0): it takes and returns real values directly.

    def predict(X) -> np.ndarray:
        probs = np.empty(len(X), dtype=np.float32)
        for i in range(len(X)):
            x = X[i].astype(np.float32)[np.newaxis, :, np.newaxis]   # (1, 6000, 1)
            if quantized:
                x = np.clip(np.round(x / in_scale) + in_zero, -128, 127).astype(np.int8)
                # Quantise the z-scored window: real value -> int8. np.clip keeps a
                # rare extreme sample inside the int8 range instead of wrapping around.
            interpreter.set_tensor(inp["index"], x)
            interpreter.invoke()
            raw = interpreter.get_tensor(out["index"])[0, 0]
            probs[i] = (int(raw) - out_zero) * out_scale if quantized else raw
            # Dequantise the int8 output: int8 -> probability. int(...) first, so
            # the subtraction cannot overflow the int8 type.
        return probs

    return predict


def scores(y: np.ndarray, prob: np.ndarray, threshold: float) -> dict:
    tn, fp, fn, tp = confusion_matrix(y.astype(int), (prob > threshold).astype(int), labels=[0, 1]).ravel()
    return {"threshold": threshold,
            "auc": float(roc_auc_score(y, prob)),
            "accuracy": float((tp + tn) / len(y)),
            "sensitivity": float(tp / (tp + fn)),               # share of apnea minutes detected
            "specificity": float(tn / (tn + fp)),               # share of normal minutes left alone
            "precision": float(tp / (tp + fp)) if tp + fp else 0.0,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}


print(f"test set: {len(nights_te)} people, {len(y_te)} minutes, {y_te.mean():.1%} apnea, "
      f"{SMOOTH_WINDOW}-minute smoothing, threshold chosen on validation\n")
print(f"{'model':7s} {'threshold':>9s} {'AUC':>7s} {'accuracy':>9s} {'sensitivity':>12s} "
      f"{'specificity':>12s} {'precision':>10s}")
results = {}
for name, predict in (("Keras", predict_keras), ("float32", tflite_predictor(FLOAT32_PATH)),
                      ("int8", tflite_predictor(TFLITE_PATH))):
    prob_val = smooth(predict(X_val), night_val)
    threshold = float(max(THRESHOLD_GRID, key=lambda t: np.mean((prob_val > t) == y_val)))
    # The threshold that maximises accuracy on validation. It is picked there,
    # never on test: a threshold tuned on the test set would make the test
    # score optimistic. Each model gets its own, since quantisation shifts the
    # probabilities.
    prob_te = smooth(predict(X_te), night_te)
    results[name] = s = scores(y_te, prob_te, threshold)
    print(f"{name:7s} {threshold:9.2f} {s['auc']:7.4f} {s['accuracy']:9.2%} {s['sensitivity']:12.3f} "
          f"{s['specificity']:12.3f} {s['precision']:10.3f}")
TEST_PATH.write_text(json.dumps(results, indent=1))

for name, s in results.items():
    print(f"\n{name + ' confusion matrix':26s}{'predicted N':>11s}  {'predicted A':>11s}")
    print(f"      true N              {s['tn']:11d}  {s['fp']:11d}")
    print(f"      true A              {s['fn']:11d}  {s['tp']:11d}")
print(f"\nsaved {TEST_PATH}")